In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from imblearn.over_sampling import SMOTE
import joblib
import os

RANDOM_STATE = 42
os.makedirs('outputs', exist_ok=True)

print('All imports successful.')

All imports successful.


In [3]:
# Load cleaned CKD dataset
ckd_df = pd.read_csv('../data/chronic_kidney_disease/ckd_cleaned.csv')

print('Dataset loaded successfully')
print('Shape:', ckd_df.shape)
print('\nColumn names:')
print(ckd_df.columns.tolist())
ckd_df.head()

Dataset loaded successfully
Shape: (400, 25)

Column names:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,0.0,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,0.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,0.0,0.0,0.0,0.0,121.0,...,38.0,6000.0,4.8,0.0,0.0,0.0,0.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,4.8,0.0,1.0,0.0,1.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,1.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,0.0,0.0,0.0,1


In [4]:
# Quick sanity check — missing values and data types
print('Missing values:')
print(ckd_df.isnull().sum())
print('\nData types:')
print(ckd_df.dtypes)
print('\nTarget distribution (classification):')
print(ckd_df['classification'].value_counts())

Missing values:
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hemo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

Data types:
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc               float64
pc                float64
pcc               float64
ba                float64
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc              

In [5]:
# Separate features (X) and target (y)
X = ckd_df.drop(columns=['classification'])
y = ckd_df['classification']  # 0 = not CKD, 1 = CKD

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('\nClass distribution:')
print(y.value_counts())
print(f'\nClass imbalance ratio — Not CKD : CKD = {y.value_counts()[0]}:{y.value_counts()[1]}')

Features shape: (400, 24)
Target shape: (400,)

Class distribution:
classification
1    250
0    150
Name: count, dtype: int64

Class imbalance ratio — Not CKD : CKD = 150:250


In [6]:
# Train-test split (stratified to maintain class distribution)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # keeps 250:150 ratio in both splits
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print('\nTrain class distribution:')
print(y_train.value_counts())
print('\nTest class distribution:')
print(y_test.value_counts())

Train : 320 samples
Test  : 80 samples

Train class distribution:
classification
1    200
0    120
Name: count, dtype: int64

Test class distribution:
classification
1    50
0    30
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on train — transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Keep as DataFrames so feature names survive into importance plots
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('Scaling done (fit on train only — no leakage).')
print('\nTrain scaled stats (mean ≈ 0, std ≈ 1):')
print(X_train_scaled.describe().loc[['mean', 'std']].round(3))

Scaling done (fit on train only — no leakage).

Train scaled stats (mean ≈ 0, std ≈ 1):
        age     bp     sg     al     su    rbc     pc    pcc     ba    bgr  \
mean -0.000  0.000  0.000  0.000 -0.000  0.000  0.000  0.000  0.000 -0.000   
std   1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002   

      ...   hemo    pcv     wc     rc    htn     dm    cad  appet     pe  \
mean  ... -0.000  0.000  0.000 -0.000  0.000  0.000  0.000 -0.000 -0.000   
std   ...  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002  1.002   

        ane  
mean  0.000  
std   1.002  

[2 rows x 24 columns]


In [8]:
#Handle class imbalance with SMOTE
import collections
from imblearn.over_sampling import SMOTE

print('Class distribution BEFORE SMOTE:')
print(collections.Counter(y_train))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('\nClass distribution AFTER SMOTE:')
print(collections.Counter(y_train_res))
print(f'\nResampled train size: {X_train_res.shape[0]} samples')

Class distribution BEFORE SMOTE:
Counter({1: 200, 0: 120})

Class distribution AFTER SMOTE:
Counter({1: 200, 0: 200})

Resampled train size: 400 samples


In [9]:
#Cross-validation setup
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import cohen_kappa_score, make_scorer

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
kappa_scorer = make_scorer(cohen_kappa_score)

def cv_report(model, X, y, label):
    """Run 5-fold CV and print Accuracy / ROC-AUC / Kappa."""
    acc   = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    roc   = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    kappa = cross_val_score(model, X, y, cv=skf, scoring=kappa_scorer)
    print(f'\n[{label}]  5-Fold CV')
    print(f'  Accuracy : {acc.mean():.4f}  ±  {acc.std():.4f}')
    print(f'  ROC-AUC  : {roc.mean():.4f}  ±  {roc.std():.4f}')
    print(f'  Kappa    : {kappa.mean():.4f}  ±  {kappa.std():.4f}  ← KEY')
    return {'Accuracy': acc.mean(), 'ROC-AUC': roc.mean(), 'Kappa': kappa.mean()}

cv_results  = {}
model_names = ['Logistic Regression', 'Random Forest', 'XGBoost']
print('CV setup ready.')

CV setup ready.


# Model 1: Logistic Regression

In [10]:
#Hyperparameter tuning 

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

lr_param_grid = {
    'C'      : [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear']
}

lr_grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    lr_param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
lr_grid.fit(X_train_res, y_train_res)

print('Best LR params :', lr_grid.best_params_)
print('Best CV ROC-AUC:', round(lr_grid.best_score_, 4))


Best LR params : {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV ROC-AUC: 1.0


In [11]:
# Train best LR on full resampled train set

lr_best = lr_grid.best_estimator_
lr_best.fit(X_train_res, y_train_res)

cv_results['Logistic Regression'] = cv_report(lr_best, X_train_res, y_train_res, 'Logistic Regression')


[Logistic Regression]  5-Fold CV
  Accuracy : 0.9100  ±  0.0146
  ROC-AUC  : 1.0000  ±  0.0000
  Kappa    : 0.8200  ±  0.0292  ← KEY


# Model 2 - Random Forest

In [12]:
#Hyperparameter tuning with GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_param_grid = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2],
    'max_features'     : ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    rf_param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_train_res, y_train_res)

print('Best RF params :', rf_grid.best_params_)
print('Best CV ROC-AUC:', round(rf_grid.best_score_, 4))

Best RF params : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV ROC-AUC: 0.9999


In [13]:
# Train best RF on full resampled train set
rf_best = rf_grid.best_estimator_
rf_best.fit(X_train_res, y_train_res)

cv_results['Random Forest'] = cv_report(rf_best, X_train_res, y_train_res, 'Random Forest')


[Random Forest]  5-Fold CV
  Accuracy : 0.9925  ±  0.0061
  ROC-AUC  : 0.9999  ±  0.0003
  Kappa    : 0.9850  ±  0.0122  ← KEY


# Model 3 - XGBoost

In [14]:
# Hyperparameter tuning with GridSearchCV
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# scale_pos_weight = majority / minority (on original train, before SMOTE)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = round(neg / pos, 4)
print(f'scale_pos_weight = {spw}  (neg={neg}, pos={pos})')

xgb_param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'subsample'       : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'reg_alpha'       : [0, 0.1, 0.5],
    'reg_lambda'      : [1, 1.5, 2]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=spw,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=RANDOM_STATE
    ),
    xgb_param_grid,
    n_iter=30,
    cv=skf,
    scoring='roc_auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)
xgb_search.fit(X_train_res, y_train_res)

print('Best XGB params :', xgb_search.best_params_)
print('Best CV ROC-AUC :', round(xgb_search.best_score_, 4))

scale_pos_weight = 0.6  (neg=120, pos=200)
Best XGB params : {'subsample': 1.0, 'reg_lambda': 1.5, 'reg_alpha': 0, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
Best CV ROC-AUC : 0.9998


In [15]:
xgb_best = xgb_search.best_estimator_
xgb_best.fit(X_train_res, y_train_res)

cv_results['XGBoost'] = cv_report(xgb_best, X_train_res, y_train_res, 'XGBoost')


[XGBoost]  5-Fold CV
  Accuracy : 0.9850  ±  0.0146
  ROC-AUC  : 0.9998  ±  0.0005
  Kappa    : 0.9700  ±  0.0292  ← KEY
